<a href="https://colab.research.google.com/github/irullah/comic-yolo/blob/main/Auto_Annotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q ultralytics

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import cv2
import shutil
import zipfile
import numpy as np
from ultralytics import YOLO
from google.colab import files

# Fungsi pembantu membaca/menyimpan gambar dengan aman
def read_image_utf8(path):
    img_array = np.fromfile(path, np.uint8)
    return cv2.imdecode(img_array, cv2.IMREAD_COLOR)

def write_image_utf8(path, img):
    ext = os.path.splitext(path)[1]
    is_success, im_buf_arr = cv2.imencode(ext, img)
    if is_success:
        im_buf_arr.tofile(path)

# ==========================================
# 1. SETUP FOLDER
# ==========================================
input_dir = "dataset_mentah"
output_dir = "siap_roboflow"

if os.path.exists(input_dir): shutil.rmtree(input_dir)
if os.path.exists(output_dir): shutil.rmtree(output_dir)

os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# ==========================================
# 2. EKSTRAK ZIP GAMBAR MENTAH
# ==========================================
zip_candidates = [f for f in os.listdir('.') if f.lower().endswith('.zip') and not f.startswith('Siap_Upload')]

if not zip_candidates:
    print("❌ ERROR: Upload dulu file ZIP berisi gambar mentah ke panel kiri Colab!")
else:
    zip_path = zip_candidates[0]
    print(f"📦 Mengekstrak {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(input_dir)

    valid_extensions = ('.jpg', '.jpeg', '.png', '.webp')
    image_files = []
    for root, dirs, files_in_dir in os.walk(input_dir):
        for file in files_in_dir:
            if file.lower().endswith(valid_extensions):
                image_files.append(os.path.join(root, file))

    image_files.sort()
    total_raw = len(image_files)

    if total_raw == 0:
        print("❌ ERROR: Tidak ditemukan file gambar di dalam ZIP!")
    else:
        # ==========================================
        # 3. SLICING + AUTO-ANNOTATION LANGSUNG KE TXT
        # ==========================================
        print(f"🤖 Memuat model dan memproses {total_raw} gambar mentah...")
        model_path = "/content/drive/MyDrive/comic cleaner/best.pt"
        model = YOLO(model_path)

        total_slices = 0
        total_labeled_slices = 0

        for idx, img_path in enumerate(image_files, 1):
            img = read_image_utf8(img_path)
            if img is None:
                continue

            H, W = img.shape[:2]
            slice_h = W
            overlap = int(slice_h * 0.1)
            step = slice_h - overlap

            base_name = os.path.splitext(os.path.basename(img_path))[0]
            slice_num = 1

            for y in range(0, H, step):
                y1 = y
                y2 = min(y + slice_h, H)
                if y2 - y1 < 50:
                    break

                crop_img = img[y1:y2, 0:W]
                total_slices += 1

                # Nama unik untuk setiap potongan agar tidak saling menimpa
                slice_filename = f"img{idx:03d}_{base_name}_s{slice_num:02d}"
                save_img_path = os.path.join(output_dir, f"{slice_filename}.jpg")
                save_txt_path = os.path.join(output_dir, f"{slice_filename}.txt")

                # Simpan potongan gambar (.jpg)
                write_image_utf8(save_img_path, crop_img)

                # Jalankan deteksi YOLO pada potongan gambar
                results = model.predict(crop_img, conf=0.10, verbose=False)[0]

                # Jika ada teks yang terdeteksi, tulis koordinat poligon langsung ke file .txt
                if results.masks is not None and len(results.masks.xyn) > 0:
                    total_labeled_slices += 1
                    with open(save_txt_path, "w") as f:
                        for cls_id, poly_norm in zip(results.boxes.cls, results.masks.xyn):
                            # Minimal 3 titik untuk membentuk poligon yang valid di Roboflow
                            if len(poly_norm) >= 3:
                                cls_int = int(cls_id.item())
                                coords_str = " ".join([f"{pt[0]:.6f} {pt[1]:.6f}" for pt in poly_norm])
                                f.write(f"{cls_int} {coords_str}\n")

                slice_num += 1

            print(f"✅ [{idx}/{total_raw}] {os.path.basename(img_path)} selesai dipotong & dilabeli.")

        # ==========================================
        # 4. RINGKASAN & DOWNLOAD ZIP
        # ==========================================
        print("\n==========================================")
        print(f"📊 RINGKASAN AUTO-ANNOTATION:")
        print(f"   - Gambar Mentah       : {total_raw} file")
        print(f"   - Total Potongan      : {total_slices} gambar (.jpg)")
        print(f"   - Potongan Berlabel   : {total_labeled_slices} file (.txt)")
        print(f"   - Potongan Tanpa Teks : {total_slices - total_labeled_slices} gambar")
        print("==========================================\n")

        output_zip = "roboflow"
        shutil.make_archive(output_zip, 'zip', output_dir)

        print(f"🎉 Mengunduh {output_zip}.zip...")
        files.download(f"{output_zip}.zip")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
📦 Mengekstrak dataset.zip...
🤖 Memuat model dan memproses 125 gambar mentah...
✅ [1/125] 01 (10).jpg selesai dipotong & dilabeli.
✅ [2/125] 01 (2).jpg selesai dipotong & dilabeli.
✅ [3/125] 01 (3).jpg selesai dipotong & dilabeli.
✅ [4/125] 01 (4).jpg selesai dipotong & dilabeli.
✅ [5/125] 01 (5).jpg selesai dipotong & dilabeli.
✅ [6/125] 01 (6).jpg selesai dipotong & dilabeli.
✅ [7/125] 01 (7).jpg selesai dipotong & dilabeli.
✅ [8/125] 01 (8).jpg selesai dipotong & dilabeli.
✅ [9/125] 01 (9).jpg selesai dipotong & dilabeli.
✅ [10/125] 01.jpg selesai dipotong & dilabeli.
✅ [11/125] 02 (10).jpg selesai dipotong & dilabeli.
✅ [12/125] 02 (2).jpg selesai dipotong & dilabeli.
✅ [13/125] 02 (3).jpg seles

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>